In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import numpy as np
from train_model import LipReadingModel  # Import previous model class
from video_processing import VideoLipProcessor  # Import video processing

def fine_tune_model(model_
                    path, new_dataset_path, old_classes):
    """Fine-tunes an existing model with new classes"""
    
    processor = VideoLipProcessor()
    
    # Load existing model
    model = load_model(model_path)
    
    # Extract existing layers
    base_model = Model(inputs=model.input, outputs=model.layers[-3].output)  # Keep up to last LSTM layer
    
    # Freeze all layers except the last ones
    for layer in base_model.layers:
        layer.trainable = False  # Prevent overfitting
    
    # Get new dataset
    (X_new, y_new), new_classes = prepare_dataset(new_dataset_path)  # Process new dataset
    updated_classes = old_classes + new_classes  # Merge old & new vocabulary
    
    num_new_classes = len(updated_classes)
    
    # Create new output layer
    x = Dropout(0.5)(base_model.output)
    new_output = Dense(num_new_classes, activation="softmax")(x)
    
    # Create new model
    new_model = Model(inputs=base_model.input, outputs=new_output)
    
    # Compile with lower learning rate for fine-tuning
    new_model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    # Split new dataset
    X_train, X_val, y_train, y_val = train_test_split(X_new, y_new, test_size=0.2, random_state=42)
    
    # Train only on new data
    new_model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=16,
        verbose=1
    )
    
    # Save updated model
    new_model.save("updated_lipreading_model.h5")
    
    return updated_classes  # Return new vocabulary

if __name__ == "__main__":
    old_vocab = ["hello", "goodbye", "yes", "no", "thank you"]  # Existing classes
    new_dataset_path = "path_to_new_dataset"  # Update with real path
    updated_vocab = fine_tune_model("best_lipreading_model.h5", new_dataset_path, old_vocab)
    
    print("Updated vocabulary:", updated_vocab)


SyntaxError: invalid syntax (3341143437.py, line 11)